# Fine-tuning BGE-M3 for African Health QA Retrieval

This notebook fine-tunes `BAAI/bge-m3` on a query-to-query retrieval task for health QA in low-resource African languages (Akan, Luganda, Swahili, Amharic + English variants).

**Setup:**
- **Task:** Given a query, retrieve the most similar train query whose answer best matches the gold answer (measured by ROUGE-1).
- **Approach:** LoRA fine-tuning with `MultipleNegativesRankingLoss` on ROUGE-1-derived triplets.
- **Target hardware:** RTX 4050 (6GB VRAM) — uses fp16, gradient checkpointing, small batch size.
- **Evaluation:** per-subset ROUGE-1 on held-out validation split.

**Pipeline:**
1. Load data → stratified train/val split per subset
2. Baseline: embed with frozen BGE-M3 → top-1 retrieval → per-subset ROUGE-1
3. Mine triplets `(anchor_query, positive_query, hard_negative_query)` using ROUGE-1 on answers
4. LoRA fine-tune
5. Re-evaluate per subset and compare

## 1. Install dependencies

Run once. Skip if already installed.

In [ ]:
# Install dependencies (uncomment when running for the first time)
# !pip install -q -U sentence-transformers>=3.0 peft transformers datasets accelerate
# !pip install -q rouge-score pandas scikit-learn tqdm matplotlib

## 2. Imports & config

In [ ]:
import os
import gc
import random
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from rouge_score import rouge_scorer
import matplotlib.pyplot as plt

from sentence_transformers import SentenceTransformer, losses
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.trainer import SentenceTransformerTrainer
from datasets import Dataset
from peft import LoraConfig, TaskType

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
CONFIG = {
    # Data
    'data_path': 'Train.csv',
    'val_ratio': 0.10,

    # Model
    'model_name': 'BAAI/bge-m3',
    'max_seq_length': 256,         # queries are short; saves a lot of VRAM
    'output_dir': './bge-m3-health-qa',

    # Triplet mining
    'top_k_retrieval': 20,         # candidates per anchor
    'positive_rouge_threshold': 0.45,  # tune after looking at distribution
    'negative_rouge_threshold': 0.15,
    'max_positives_per_anchor': 2,
    'max_negatives_per_anchor': 3,
    'mine_within_subset_only': True,   # ROUGE across scripts is meaningless

    # Training (tuned for 6GB VRAM)
    'batch_size': 4,
    'gradient_accumulation_steps': 8,  # effective batch = 32
    'learning_rate': 2e-5,
    'num_epochs': 2,
    'warmup_ratio': 0.1,
    'fp16': True,
    'gradient_checkpointing': True,

    # LoRA
    'lora_r': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.1,
    'lora_target_modules': ['query', 'key', 'value', 'dense'],

    # Encoding
    'encode_batch_size': 32,
}

## 3. Load and split data

Stratified split per subset so every subset is represented in val.

In [ ]:
df = pd.read_csv(CONFIG['data_path'])
df = df.dropna(subset=['input', 'output']).reset_index(drop=True)
print(f"Total rows: {len(df)}")
print(f"\nSubset distribution:")
print(df['subset'].value_counts())

In [ ]:
train_df, val_df = train_test_split(
    df,
    test_size=CONFIG['val_ratio'],
    stratify=df['subset'],
    random_state=SEED,
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)}")
print(f"\nVal subset distribution:")
print(val_df['subset'].value_counts())

## 4. ROUGE-1 scorer

For African languages we **don't** use the English stemmer (it would break tokens in Amharic, mangle Akan etc.).

In [ ]:
scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=False)

def rouge1(reference: str, candidate: str) -> float:
    if not reference or not candidate:
        return 0.0
    return scorer.score(reference, candidate)['rouge1'].fmeasure

## 5. Retrieval evaluation function

For each val query:
1. Embed it
2. Find the most similar train query (top-1) by cosine similarity
3. Return that train item's **answer**
4. Compute ROUGE-1 against the val gold answer

This mirrors your actual use case. We also support `top_k > 1` for oracle evaluation.

In [ ]:
@torch.no_grad()
def evaluate_retrieval(model, train_df, val_df, top_k=1, encode_batch_size=32):
    """Top-1 cosine retrieval (or oracle@k if top_k > 1).
    Returns (per_subset_df, overall_mean, raw_results_df)."""
    model.eval()
    train_queries = train_df['input'].tolist()
    train_answers = train_df['output'].tolist()
    val_queries = val_df['input'].tolist()
    val_answers = val_df['output'].tolist()
    val_subsets = val_df['subset'].tolist()

    print(f"Encoding {len(train_queries)} train queries...")
    train_embs = model.encode(
        train_queries, batch_size=encode_batch_size, show_progress_bar=True,
        convert_to_numpy=True, normalize_embeddings=True,
    )
    print(f"Encoding {len(val_queries)} val queries...")
    val_embs = model.encode(
        val_queries, batch_size=encode_batch_size, show_progress_bar=True,
        convert_to_numpy=True, normalize_embeddings=True,
    )

    rows = []
    chunk = 256
    for start in tqdm(range(0, len(val_embs), chunk), desc='Scoring'):
        end = min(start + chunk, len(val_embs))
        sims = val_embs[start:end] @ train_embs.T  # (chunk, N_train)
        if top_k == 1:
            top_idx = sims.argmax(axis=1)
            for i_local, j in enumerate(top_idx):
                i_global = start + i_local
                score = rouge1(val_answers[i_global], train_answers[j])
                rows.append({'subset': val_subsets[i_global], 'rouge1': score})
        else:
            top_idx = np.argsort(-sims, axis=1)[:, :top_k]
            for i_local in range(end - start):
                i_global = start + i_local
                best = max(
                    rouge1(val_answers[i_global], train_answers[j])
                    for j in top_idx[i_local]
                )
                rows.append({'subset': val_subsets[i_global], 'rouge1': best})

    res_df = pd.DataFrame(rows)
    per_subset = res_df.groupby('subset')['rouge1'].agg(['mean', 'std', 'count']).round(4)
    overall = res_df['rouge1'].mean()
    return per_subset, overall, res_df

## 6. Baseline: frozen BGE-M3

Get the numbers we need to beat. Also compute oracle@20 as the upper bound — the gap is your headroom.

In [ ]:
print("Loading baseline BGE-M3...")
baseline_model = SentenceTransformer(CONFIG['model_name'])
baseline_model.max_seq_length = CONFIG['max_seq_length']

In [ ]:
# Top-1 baseline
print("=== Baseline top-1 retrieval ===")
baseline_top1, baseline_overall, _ = evaluate_retrieval(
    baseline_model, train_df, val_df, top_k=1,
    encode_batch_size=CONFIG['encode_batch_size'],
)
print(f"\nOverall ROUGE-1 (top-1): {baseline_overall:.4f}")
print(baseline_top1)

In [ ]:
# Oracle@20 — upper bound if we could perfectly rerank within the top 20
print("=== Oracle @ top-20 ===")
oracle_top20, oracle_overall, _ = evaluate_retrieval(
    baseline_model, train_df, val_df, top_k=20,
    encode_batch_size=CONFIG['encode_batch_size'],
)
print(f"\nOverall ROUGE-1 (oracle@20): {oracle_overall:.4f}")
print(oracle_top20)

In [ ]:
# Headroom: top-1 vs oracle@20
headroom = pd.DataFrame({
    'top1': baseline_top1['mean'],
    'oracle@20': oracle_top20['mean'],
})
headroom['headroom'] = (headroom['oracle@20'] - headroom['top1']).round(4)
print('Headroom per subset (how much fine-tuning could close):')
print(headroom.sort_values('headroom', ascending=False))

## 7. Build training triplets

For each anchor query, find its top-20 neighbors *within the same subset* (cross-script ROUGE is noise), compute ROUGE-1 between answers, and pick:
- **Positives:** ROUGE-1 ≥ 0.45 — different surface form, similar answer
- **Hard negatives:** ROUGE-1 ≤ 0.15 — model thinks they're similar but answer is wrong

Inspect the ROUGE distribution first to confirm thresholds.

In [ ]:
# Pre-compute training embeddings (re-used for mining)
print("Encoding train queries for triplet mining...")
train_query_embs = baseline_model.encode(
    train_df['input'].tolist(),
    batch_size=CONFIG['encode_batch_size'],
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print(f"Shape: {train_query_embs.shape}")

In [ ]:
# Sanity check: distribution of ROUGE-1 over top-20 (within subset, excluding self)
sample_size = min(1000, len(train_df))
sample_idx = np.random.choice(len(train_df), sample_size, replace=False)
subsets_arr = train_df['subset'].values
answers_arr = train_df['output'].values

rouge_samples = []
for i in tqdm(sample_idx, desc='Sampling ROUGE distribution'):
    same_subset = np.where((subsets_arr == subsets_arr[i]) & (np.arange(len(subsets_arr)) != i))[0]
    if len(same_subset) == 0:
        continue
    sims = train_query_embs[i] @ train_query_embs[same_subset].T
    top = same_subset[np.argsort(-sims)[:CONFIG['top_k_retrieval']]]
    for j in top:
        rouge_samples.append(rouge1(answers_arr[i], answers_arr[j]))

plt.figure(figsize=(8, 4))
plt.hist(rouge_samples, bins=50, edgecolor='black')
plt.axvline(CONFIG['positive_rouge_threshold'], color='green', linestyle='--', label=f"pos ≥ {CONFIG['positive_rouge_threshold']}")
plt.axvline(CONFIG['negative_rouge_threshold'], color='red', linestyle='--', label=f"neg ≤ {CONFIG['negative_rouge_threshold']}")
plt.xlabel('ROUGE-1 (anchor answer vs top-20 neighbor answer)')
plt.ylabel('Count')
plt.title('ROUGE-1 distribution among top-20 retrieved neighbors')
plt.legend()
plt.tight_layout()
plt.show()
print(f"Mean: {np.mean(rouge_samples):.3f} | Median: {np.median(rouge_samples):.3f}")

In [ ]:
def mine_triplets(train_df, embs, config):
    queries = train_df['input'].values
    answers = train_df['output'].values
    subsets = train_df['subset'].values
    n = len(queries)

    triplets = []
    stats = {'no_positives': 0, 'no_negatives': 0, 'ok': 0}

    chunk = 256
    for start in tqdm(range(0, n, chunk), desc='Mining triplets'):
        end = min(start + chunk, n)
        sims_chunk = embs[start:end] @ embs.T  # (chunk, n)

        for i_local, i_global in enumerate(range(start, end)):
            sims = sims_chunk[i_local].copy()
            sims[i_global] = -np.inf  # exclude self

            if config['mine_within_subset_only']:
                other_subset = subsets != subsets[i_global]
                sims[other_subset] = -np.inf

            k = min(config['top_k_retrieval'], int((sims > -np.inf).sum()))
            if k <= 0:
                continue
            top_idx = np.argpartition(-sims, k - 1)[:k]
            top_idx = top_idx[sims[top_idx] > -np.inf]

            anchor_ans = answers[i_global]
            scored = [(int(j), rouge1(anchor_ans, answers[j])) for j in top_idx]
            scored.sort(key=lambda x: -x[1])

            positives = [j for j, r in scored if r >= config['positive_rouge_threshold']]
            negatives = [j for j, r in scored if r <= config['negative_rouge_threshold']]

            positives = positives[:config['max_positives_per_anchor']]
            negatives = negatives[:config['max_negatives_per_anchor']]

            if not positives:
                stats['no_positives'] += 1
                continue
            if not negatives:
                stats['no_negatives'] += 1
                continue
            stats['ok'] += 1

            for p in positives:
                for ng in negatives:
                    triplets.append((queries[i_global], queries[p], queries[ng]))

    return triplets, stats

triplets, stats = mine_triplets(train_df, train_query_embs, CONFIG)
print(f"\nTriplets built: {len(triplets):,}")
print(f"Anchors with no positives: {stats['no_positives']}")
print(f"Anchors with no negatives: {stats['no_negatives']}")
print(f"Anchors that produced triplets: {stats['ok']}")

In [ ]:
# Sanity-check a few triplets
print("=== Sample triplets ===")
for i, (a, p, n) in enumerate(random.sample(triplets, min(3, len(triplets)))):
    print(f"\n--- Triplet {i+1} ---")
    print(f"ANCHOR:   {a[:200]}")
    print(f"POSITIVE: {p[:200]}")
    print(f"NEGATIVE: {n[:200]}")

In [ ]:
# Free baseline model before training
del baseline_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 8. LoRA fine-tuning

Settings tuned for 6GB VRAM:
- LoRA on attention + dense layers (only ~1% of params are trainable)
- fp16 + gradient checkpointing
- batch size 4 with 8x accumulation → effective batch 32
- `MultipleNegativesRankingLoss` uses our mined hard negative **plus** in-batch negatives — very sample-efficient

In [ ]:
# Load fresh model and attach LoRA
model = SentenceTransformer(CONFIG['model_name'])
model.max_seq_length = CONFIG['max_seq_length']

lora_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    inference_mode=False,
    r=CONFIG['lora_r'],
    lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'],
    target_modules=CONFIG['lora_target_modules'],
    bias='none',
)
model.add_adapter(lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
# Build a HF Dataset of triplets
train_ds = Dataset.from_dict({
    'anchor':   [t[0] for t in triplets],
    'positive': [t[1] for t in triplets],
    'negative': [t[2] for t in triplets],
})
train_ds = train_ds.shuffle(seed=SEED)
print(train_ds)

In [ ]:
train_loss = losses.MultipleNegativesRankingLoss(model=model)

args = SentenceTransformerTrainingArguments(
    output_dir=CONFIG['output_dir'],
    num_train_epochs=CONFIG['num_epochs'],
    per_device_train_batch_size=CONFIG['batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
    learning_rate=CONFIG['learning_rate'],
    warmup_ratio=CONFIG['warmup_ratio'],
    fp16=CONFIG['fp16'],
    gradient_checkpointing=CONFIG['gradient_checkpointing'],
    logging_steps=50,
    save_strategy='epoch',
    save_total_limit=1,
    report_to='none',
    dataloader_num_workers=2,
    remove_unused_columns=False,
    # If your GPU supports bf16 (Ada Lovelace does — RTX 4050 included), it's more stable:
    # bf16=True, fp16=False,
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    loss=train_loss,
)

In [ ]:
trainer.train()

In [ ]:
# Save the LoRA-adapted model
model.save_pretrained(CONFIG['output_dir'] + '/final')
print(f"Saved to: {CONFIG['output_dir']}/final")

## 9. Evaluate the fine-tuned model

In [ ]:
print("=== Fine-tuned top-1 retrieval ===")
ft_top1, ft_overall, _ = evaluate_retrieval(
    model, train_df, val_df, top_k=1,
    encode_batch_size=CONFIG['encode_batch_size'],
)
print(f"\nOverall ROUGE-1 (top-1, fine-tuned): {ft_overall:.4f}")
print(ft_top1)

In [ ]:
print("=== Fine-tuned oracle @ top-20 ===")
ft_oracle, ft_oracle_overall, _ = evaluate_retrieval(
    model, train_df, val_df, top_k=20,
    encode_batch_size=CONFIG['encode_batch_size'],
)
print(f"\nOverall oracle@20 (fine-tuned): {ft_oracle_overall:.4f}")
print(ft_oracle)

## 10. Compare baseline vs fine-tuned

In [ ]:
comparison = pd.DataFrame({
    'baseline_top1':   baseline_top1['mean'],
    'finetuned_top1':  ft_top1['mean'],
    'oracle@20':       oracle_top20['mean'],
    'ft_oracle@20':    ft_oracle['mean'],
    'val_count':       baseline_top1['count'].astype(int),
})
comparison['delta_top1'] = (comparison['finetuned_top1'] - comparison['baseline_top1']).round(4)
comparison['headroom_closed'] = (
    (comparison['finetuned_top1'] - comparison['baseline_top1']) /
    (comparison['oracle@20'] - comparison['baseline_top1']).replace(0, np.nan)
).round(3)
comparison = comparison.sort_values('delta_top1', ascending=False)
print(comparison)

In [ ]:
# Visualize per-subset improvement
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(comparison))
w = 0.25
ax.bar(x - w, comparison['baseline_top1'], w, label='Baseline top-1', color='#888888')
ax.bar(x,     comparison['finetuned_top1'], w, label='Fine-tuned top-1', color='#2a9d8f')
ax.bar(x + w, comparison['oracle@20'],     w, label='Oracle @ top-20', color='#e9c46a')
ax.set_xticks(x)
ax.set_xticklabels(comparison.index, rotation=30, ha='right')
ax.set_ylabel('ROUGE-1')
ax.set_title('Per-subset retrieval quality')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print("OVERALL ROUGE-1")
print(f"  Baseline top-1:    {baseline_overall:.4f}")
print(f"  Fine-tuned top-1:  {ft_overall:.4f}   (delta = {ft_overall - baseline_overall:+.4f})")
print(f"  Oracle @ top-20:   {oracle_overall:.4f}")
denom = max(oracle_overall - baseline_overall, 1e-9)
print(f"  Headroom closed:   {(ft_overall - baseline_overall) / denom * 100:.1f}%")

## 11. Tips & next steps

**If results are disappointing per subset:**
- Inspect the **ROUGE distribution plot**. If the curve is flat (no clear positives/negatives), threshold gating drops too many anchors. Lower `positive_rouge_threshold` (try 0.35) or raise `negative_rouge_threshold` (try 0.20).
- Check `stats['no_positives']`. If >50% of anchors have none, your dataset has few near-duplicates and contrastive learning will struggle. Consider self-positives via dropout augmentation (SimCSE-style) or query paraphrasing with an LLM.
- **Per-subset learning:** if Amharic/Akan regress while Swahili improves, do multi-stage training — first all subsets, then per-language fine-tunes with lower LR.

**If you hit OOM:**
- Drop `max_seq_length` to 192 or 128
- `batch_size=2`, `gradient_accumulation_steps=16`
- Remove `dense` from `lora_target_modules` (keep attention only)
- Try `bf16=True, fp16=False` — RTX 4050 (Ada) supports bf16, often more stable than fp16

**To push further:**
- Train a tiny ROUGE-1 predictor on `(query_emb, candidate_emb, BM25_score) → ROUGE-1` and use it as a learned reranker over the top-20 — directly mimicking your oracle.
- Ensemble fine-tuned BGE-M3 with an AfroXLMR-trained encoder, especially for Akan and Luganda where XLM-R's pretraining is weakest.